In [0]:
SELECT * FROM users;

### Q1. Write a SQL query to find duplicate records in a table 

In [0]:
-- Solution 1 for a summary of who is duplicated
SELECT name,email 
FROM users
GROUP BY name,email
HAVING count(*) > 1;

-- Solution 2 for identifying every single redundant row
SELECT * 
FROM (SELECT *,row_number() over(PARTITION BY name,email ORDER BY user_id) AS ranked FROM users) 
WHERE ranked > 1;

In [0]:
-- We can't just do DELETE FROM (subquery) WHERE ranked > 1 directly in standard SQL dialects (like SQL Server, MySQL, or PostgreSQL) because databases don't always allow deleting directly from a windowed subquery.

DELETE
FROM (SELECT *,row_number() over(PARTITION BY name,email ORDER BY name,email) AS ranked FROM users)
WHERE ranked > 1;

In [0]:
WITH CTE_Duplicates AS (
  SELECT
    *,
    row_number() over (PARTITION BY name, email ORDER BY name, email) AS ranked
  FROM
    users
)
DELETE FROM CTE_Duplicates
WHERE ranked > 1;

---
#### The Difference from Cell `Using Subqueries` & `Using CTEs`
- Cell 4 failed with a syntax error because you can't DELETE directly from a subquery in the FROM clause.
- Cell 5 is syntactically valid (CTEs with DELETE are legal SQL in many databases), but fails at Databricks's security verification layer because the permission checker cannot validate DELETE rights on a temporary CTE.

In [0]:
DELETE FROM users
WHERE user_id IN (
  SELECT user_id
    FROM (
        SELECT
        user_id,
        row_number() OVER (PARTITION BY name, email ORDER BY user_id) AS ranked
        FROM users
    ) tbl
  WHERE ranked > 1
);

In [0]:
SELECT * FROM users;

In [0]:
DESCRIBE HISTORY users;

Here are the **Databricks-specific ways to delete duplicates**:

## ✅ Approaches That Work in Databricks

### 1. DELETE with WHERE IN + Subquery (Cell 7 - already working)
Uses a subquery to identify duplicate `user_id` values and deletes them. This is the most straightforward approach when you have a unique identifier.

### 2. MERGE Statement (Delta Lake feature)
```sql
MERGE INTO users AS target
USING (
  SELECT user_id
  FROM (
    SELECT user_id, 
           ROW_NUMBER() OVER (PARTITION BY name, email ORDER BY user_id) as rn
    FROM users
  )
  WHERE rn > 1
) AS source
ON target.user_id = source.user_id
WHEN MATCHED THEN DELETE;
```
Keeps only rows where `rn = 1`, deletes everything else.

### 3. CREATE OR REPLACE TABLE
```sql
CREATE OR REPLACE TABLE users AS
SELECT user_id, name, email, signup_date
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY name, email ORDER BY user_id) as rn
  FROM users
)
WHERE rn = 1
```
Completely rewrites the table with deduplicated data. Fast for large-scale deduplication.

### 4. INSERT OVERWRITE
```sql
INSERT OVERWRITE users
SELECT user_id, name, email, signup_date
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY name, email ORDER BY user_id) as rn
  FROM users
)
WHERE rn = 1
```
Replaces all data in the table with the deduplicated result.

### 5. QUALIFY Clause (Databricks SQL feature)
```sql
DELETE FROM users
WHERE user_id NOT IN (
  SELECT user_id
  FROM users
  QUALIFY ROW_NUMBER() OVER (PARTITION BY name, email ORDER BY user_id) = 1
)
```
`QUALIFY` filters rows directly based on window function results - cleaner syntax than nested subqueries.

### 6. Delta MERGE with Temp Table
Create a temp table with deduplicated data, then merge:
```sql
CREATE OR REPLACE TEMP VIEW users_deduped AS
SELECT * FROM users
QUALIFY ROW_NUMBER() OVER (PARTITION BY name, email ORDER BY user_id) = 1;

MERGE INTO users
USING users_deduped
ON users.user_id = users_deduped.user_id
WHEN NOT MATCHED BY SOURCE THEN DELETE
```

## 🚀 Why Interviewers Love `QUALIFY`, `INSERT OVERWRITE`, and `MERGE`

1. **The `QUALIFY` Clause:** This is a major flex in a Databricks interview. In standard SQL, you are forced to write a nested subquery just to filter on a window function because you can't use window calculations in a `WHERE` clause. `QUALIFY` does for window functions what `HAVING` does for `GROUP BY`. It makes your code incredibly clean.
2. **`CREATE OR REPLACE TABLE` (CTAS) & `INSERT OVERWRITE`:** In a distributed big data environment like Spark/Delta Lake, traditional `DELETE` statements can be slow because they have to find individual rows, rewrite the underlying Parquet files, and log tombstones. Completely overwriting the table with just the `rn = 1` records is actually **massively faster** for large datasets because it performs a bulk write.
3. **`MERGE INTO` with `WHEN NOT MATCHED BY SOURCE THEN DELETE`:** This showcases that you understand how Delta Lake's ACID transaction log handles row-level mutations compared to traditional database storage.

---

## 🎯 Best Practices for Databricks

* **For small tables**: Use `DELETE WHERE IN` (Cell 7 approach)
* **For large tables**: Use `CREATE OR REPLACE TABLE` or `INSERT OVERWRITE` - they're optimized for bulk operations
* **For incremental pipelines**: Use `MERGE` in a scheduled job
* **For simplicity**: Use `QUALIFY` clause - it's the cleanest Databricks-native syntax

---

## ❌ Why Cell 5 Doesn't Work

CTEs are **query-scoped temporary results**, not real tables. Databricks's permission system needs to verify DELETE access against an **actual catalog table**, but `CTE_Duplicates` only exists during query execution - there's no physical table to check permissions against. The security layer can't trace the CTE back to `users` during its verification phase.

### Q2. Write a SQL query to get the 2nd highest salary.

In [0]:
SELECT * FROM employees;

In [0]:
With ranked_tbl AS 
(
    SELECT *,
    dense_rank() OVER (ORDER BY salary desc) AS ranked
    FROM employees
)
SELECT * FROM ranked_tbl
WHERE ranked = 2;

---
### The "Optimize for Performance" Question

- If interviewer will look at your window function and ask:
"Window functions require a full shuffle/sort of the data across the cluster to calculate the rank. If this table has 500 million rows, this will be quite slow. Can you write a simpler version that doesn't use a window function at all?"

In [0]:
SELECT MAX(salary) 
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
-- Only returns the single salary column

In [0]:
SELECT DISTINCT salary 
FROM employees
ORDER BY salary DESC
LIMIT 1 OFFSET 1;
-- Only returns the single salary column

### Q3. ROW_NUMBER() vs RANK() vs DENSE_RANK() — what's the actual difference?

To document this cleanly, let's look at how these three functions handle the exact same sample data when ordering by salary descending.

| Name | Salary | `ROW_NUMBER()` | `RANK()` | `DENSE_RANK()` |
| --- | --- | --- | --- | --- |
| **Siddhant** | 95000 | 1 | 1 | 1 |
| **Priya** | 95000 | 2 | 1 | 1 |
| **Amit** | 90000 | 3 | 3 | 2 |
| **Vikram** | 70000 | 4 | 4 | 3 |

### 📝 Breakdown:

1. **`ROW_NUMBER()` (Strict & Sequential):**
* **Behavior:** Assigns a unique, consecutive integer to every single row, starting at 1.
* **Ties:** It **never allows ties**. Even though Siddhant and Priya have the same salary, one gets 1 and the other gets 2 based on whichever row Spark processes first (unless you add a second sorting column).
* **Gaps:** No gaps. (1, 2, 3, 4)
* **Production Tip:** Always add a unique tie-breaker column like `ROW_NUMBER() OVER (ORDER BY salary DESC, emp_id ASC)` to make the output predictable and deterministic.

2. **`RANK()` (Olympic Style):**
* **Behavior:** Assigns the same rank to duplicate values.
* **Ties:** Both Siddhant and Priya get rank 1.
* **Gaps:** It **skips ranks** to account for the ties. Because two people took up rank 1, rank 2 is completely skipped, and Amit jumps straight to **3**. (1, 1, 3, 4)


3. **`DENSE_RANK()` (Continuous/Dense Style):**
* **Behavior:** Assigns the same rank to duplicate values, just like `RANK()`.
* **Ties:** Both Siddhant and Priya get rank 1.
* **Gaps:** It **never leaves a gap**. The next unique value (Amit) gets the immediate next number, which is **2**. (1, 1, 2, 3)

---

### Q4. How do you optimize a slow SQL query on a large dataset?

Because We are working extensively in **Databricks**, the way you answer this should be split into two layers: **Generic SQL practices** and **Databricks/Delta Lake-specific optimizations**.

---

### 🌌 Query Optimization in Distributed Big Data (Databricks / Delta Lake)

#### 1. Optimize the Data Layout (Storage Layer)

* **Partitioning:** Ensure the table is partitioned on low-cardinality columns used frequently in filters (like `date` or `country`). This allows the engine to skip reading entire folders of irrelevant data (**Partition Pruning**).
* **Z-Ordering / Liquid Clustering (Databricks-specific):** Apply Z-Ordering or use Delta Lake's Liquid Clustering on high-cardinality filter columns (like `user_id` or `device_id`) to co-locate related data within individual Parquet files.

#### 2. Rewrite the Query Logic (Compute Layer)

* **Select Only Needed Columns:** Never use `SELECT *`. Explicitly select only the columns required for the downstream process to reduce I/O overhead and memory consumption.
* **Filter Early (`WHERE` before `JOIN`):** Apply filters as close to the source read as possible to drastically reduce the volume of data being pushed through joins and shuffles.
* **Fix Join Order & Strategies:** 
    - Ensure the larger table is on the left and the smaller table is on the right.
    - If joining a massive table with a small lookup table, broadcast the small table (**Broadcast Hash Join**) to eliminate expensive data shuffles across the cluster.



#### 3. Maintain Dataset Health (Maintenance Layer)

* **File Compaction (OPTIMIZE):** Run the `OPTIMIZE` command on your Delta tables to merge millions of tiny, scattered files into standard 1GB files, which eliminates file-listing bottlenecks.
* **Compute Statistics (ANALYZE):** Periodically run `ANALYZE TABLE COMPUTE STATISTICS` so the Spark Catalyst Optimizer has exact data layouts, enabling it to choose the fastest execution plan.

---

### 💾 Query Optimization in Relational Databases (Azure SQL Database / RDBMS)

When you shift the conversation from a big data system like **Databricks** (which processes data across a cluster of multiple servers) to a traditional relational database like **Azure SQL Database** (which runs on a single structured database engine), the optimization strategy changes completely.

In a traditional relational database, performance is all about **minimizing disk I/O, preventing table scans, and managing locks/concurrency.**

Here is the breakdown of how to optimize a slow query specifically for **Azure SQL Database** to add to your notes:

---

#### 1. Indexing (The Most Critical Layer)

* **Clustered Index:** Ensure every table has a clustered index (usually on the Primary Key). This physically dictates how the data is ordered on the disk.
* **Non-Clustered Indexes:** Create non-clustered indexes on columns frequently used in `WHERE`, `JOIN`, and `ORDER BY` clauses to avoid slow "Table Scans" and force fast "Index Seeks".
* **Covering Indexes (`INCLUDE`):** Use the `INCLUDE` clause to attach extra columns to a non-clustered index. This allows the query to get all its data directly from the index without having to do an expensive second lookup on the main table data pages (**Key Lookup**).

#### 2. Query Refactoring & SARGability

* **Write SARGable Queries:** Ensure your `WHERE` clauses are Search Argumentable (SARGable). Avoid wrapping index columns in functions.
* **Bad (Non-SARGable):** `WHERE YEAR(signup_date) = 2026` (This forces a full table scan because the engine has to compute the function for every single row).
* **Good (SARGable):** `WHERE signup_date >= '2026-01-01' AND signup_date < '2027-01-01'` (This allows the engine to jump straight to the index).


* **Avoid `OR` Conditions:** `OR` operators often break index usage. Replacing them with a `UNION ALL` combining two separate indexed queries is often much faster.

#### 3. Execution Plan & Engine Maintenance

* **Analyze the Execution Plan:** Use Azure Data Studio or SSMS to look at the graphical execution plan. Look for thick data arrows (high data volume) or warning icons indicating implicit data type conversions or missing indexes.
* **Update Statistics:** Azure SQL relies on statistics to choose the best execution path. If data changes rapidly, statistics become stale, causing the optimizer to choose poor paths. Run `UPDATE STATISTICS table_name` to refresh them.
* **Review Parameter Sniffing:** If a query runs fast in isolation but slow inside a stored procedure, use the `OPTION (RECOMPILE)` hint to force the engine to generate a fresh, optimized execution plan for that specific execution.

---

### Q5. How does indexing improve query performance & when should you use it?

This is an essential question for any traditional relational database platform (like Azure SQL). Interviewers use it to see if you actually understand storage mechanics under the hood or if you just think of an index as a "magic speed button."

---

### 🔍 How Indexing Works Under the Hood

By default, if a table has no index, the database engine has to perform a **Table Scan**—meaning it must read every single **data page** on the disk from top to bottom just to find a single row.

In a traditional relational database (like Azure SQL or SQL Server), a **data page** is the absolute smallest, fundamental unit of data storage used by the database engine to organize and read data from disk.

When you write data into a table, the database doesn't just store it as a long, continuous text file. Instead, it carves up its storage space into fixed-size blocks called **pages**, and each page is exactly **8 KB (Kilobytes)** in size.

Think of it like an actual book:

* The **Table** is the entire book.
* A **Data Page** is a single page in that book.
* The **Rows** of data are the actual lines of text printed on that page.

---

#### 🛠️ The Structure of an 8 KB Data Page

Every single data page is split into three distinct sections to keep the data organized:

```
+---------------------------------------------------+
| 1. Page Header (96 bytes)                         |
|    - Page number, Type of page, Free space, etc.  |
+---------------------------------------------------+
| 2. Actual Data Rows                               |
|    - Row 1: Siddhant | sid@example.com           |
|    - Row 2: Amit     | amit@example.com           |
|    - Row 3: Rahul    | rahul@example.com          |
+---------------------------------------------------+
| 3. Slot Array (Row Offset Table)                  |
|    - Tells the engine exactly where each row      |
|      starts inside the page (read backwards)      |
+---------------------------------------------------+

```

1. **Page Header (96 Bytes):** Stores system metadata about the page itself (like the page number, the ID of the table it belongs to, and pointers to the next and previous pages so they can be chained together).
2. **Data Rows:** This is where your actual table records are physically written. A single page can hold multiple rows, depending on how large your rows are.
3. **Slot Array (Row Offset Table):** A tiny index at the very bottom of the page that tells the database engine exactly how many bytes into the page each row starts. The engine reads this slot array backward to locate a specific row instantly.

---

#### 💡 Why Knowing This Matters to a Data Engineer

Understanding data pages completely changes how you think about **Query Optimization**:

* **Minimizing Disk I/O:** When you run a query, the database engine never reads data row-by-row directly from the hard drive. It reads *entire 8 KB pages* into its memory cache (Buffer Pool). If a query needs to look at just one row, the engine still has to pull the entire 8 KB page into memory.
* **The Danger of a Table Scan:** If your table has 1,000,000 rows spread across 50,000 pages, and you don't use an index, the engine is forced to load all 50,000 data pages into memory to find what you asked for.
* **How Indexes Help:** An index is just a specialized type of page (**Index Page**) that stores pointers. It tells the engine, *"Don't scan all 50,000 pages. The row you want is on Data Page #412."* The engine then loads exactly that *one* page, reducing disk traffic to almost zero.

An index is a separate, specialized structure (organized as a balanced tree, or **B-Tree**) that acts like an index at the back of a textbook. Instead of reading the whole book to find "Databricks," you look up the word in the alphabetical index and jump straight to the exact page number.

#### 1. Clustered Index (The Table Itself)

* **What it is:** A clustered index determines the physical order in which data is sorted and stored on the disk. Because data can only be physically sorted one way, you can have **only one** clustered index per table (typically automatically created on your Primary Key).
* **Leaf Node:** The bottom (leaf) nodes of a clustered index contain the **actual, real data rows** of the table.

#### 2. Non-Clustered Index (The Pointer/Index Book)

* **What it is:** A separate, independent structure from the actual table data. It contains a copy of a selected column (like `email`) alongside a pointer (the row identifier or clustered index key) that tells the engine exactly where the full row lives. You can have **multiple** non-clustered indexes on a table.
* **Leaf Node:** The bottom nodes contain only the indexed column value and a **pointer** to the real data page.

---

### 🎯 When Should You Use It? (The Golden Rules)

An index is a double-edged sword. While it makes reading data fast, it **slows down writes** (`INSERT`, `UPDATE`, `DELETE`) because every time data changes, the database has to update both the main table and all the separate index structures.

#### ✅ When to CREATE an index:

* **High Read-to-Write Ratio:** On tables where data is read frequently but modified rarely (e.g., product catalogs, historical lookups).
* **Frequent Filters (`WHERE`):** Columns heavily used in search criteria (e.g., `WHERE status = 'Active'`).
* **Join Keys (`JOIN`):** Foreign key columns that link tables together—this completely accelerates join operations.
* **Sorting & Grouping (`ORDER BY` / `GROUP BY`):** If an index is pre-sorted on that column, the database engine skips the expensive CPU sorting phase entirely.

#### ❌ When to AVOID an index:

* **High-Frequency Writes:** Tables experiencing massive, constant writes or streaming ingestion (e.g., transactional log tables). Extra indexes will bottleneck your system's write performance.
* **Low-Cardinality Columns:** Columns with very few unique values (e.g., a `gender` column with only 'M'/'F'). An index won't help the engine narrow down data efficiently here.
* **Small Tables:** If a table only has a few hundred rows, reading the entire table directly is actually faster than wasting time navigating an index tree structure.

---

#### 🔬 Testing Query Performance Before And After Indexing(I Perfomed Locally On MySQL)

To deeply understand the physical impact of indexing on disk I/O, a practical simulation was executed locally on MySQL using a dataset `users_raw` with **2M rows**.

---

#### 1. The Unindexed Benchmark (Full Table Scan)

A lookup query was executed to find a specific user profile by an unindexed string column (`email`).
```

-- Test 1: Find a user near the end of the table (Without Index)
SELECT * FROM users_raw WHERE email = 'user987654@example.com';
```

* **Execution Strategy:** The engine is forced to perform a complete **Table Scan**. Because the rows are unordered and lack pointers, the database loads every single 16KB storage page from the hard drive into memory, checking them row-by-row.
* **Execution Time:** 2.20 seconds 🐌
* **Rows Evaluated:** 2,000,000 rows 
* **Engine Status (`EXPLAIN` type):** ALL (In MySQL terminology, ALL means a full Table Scan.)
* **Query stats:**
> ![image_1784047948848.png](images_SQL_NB1/image_1784047948848.png "image_1784047948848.png")
* **Execution Plan:**
> ![image_1784048032992.png](images_SQL_NB1/image_1784048032992.png "image_1784048032992.png")
---

#### 2. The Optimized Benchmark (B-Tree Index Seek)

A standard Non-Clustered B-Tree index structure was applied to the search predicate column (`email`).
```

-- This creates the index structure
CREATE INDEX idx_user_email ON users_raw(email);

-- Test 2: Find a user near the end of the table (With Index)
SELECT * FROM users_raw WHERE email = 'user987654@example.com';
```

* **Execution Strategy:** The engine shifts from a scan to an **Index Seek**. It navigates a balanced tree structure of pointer pages. It matches the search term against the root node, drops down to the correct branch node, and arrives at the leaf node. The leaf node gives the engine the exact memory address of the data page it needs.
* **Execution Time:** 0.02 seconds ⚡ (*A 110x performance improvement!*)
* **Rows Evaluated:** 2 row (Bypassed 1,999,998 irrelevant records completely).
* **Engine Status (`EXPLAIN` type):** ref / const (This means it is utilizing an index pointer lookup instead of a scan.)
* **Query stats:**
> ![image_1784048629852.png](images_SQL_NB1/image_1784048629852.png "image_1784048629852.png")
* **Execution Plan:**
> ![image_1784048780395.png](images_SQL_NB1/image_1784048780395.png "image_1784048780395.png")
---

#### 3. The "Index Killer" Trap (Non-SARGable Query)

To test the limitations of a B-Tree index, a wildcard search was applied to the *beginning* of the search pattern (`LIKE '%user987654@example.com'`).
```

-- Test 3: Look for emails ending in a specific pattern using a wildcard at the START (With Index but its skiped)
SELECT * FROM users_raw WHERE email LIKE '%user987654@example.com';
```

* **Execution Strategy:** Because a B-Tree index sorts characters strictly from left-to-right, placing a wildcard at the very front makes the predicate **Non-SARGable** (Not Search Argumentable). The engine cannot look up a word by its ending characters using a forward tree, so it abandons the index entirely and regresses back to a full table scan.
* **Execution Time:** 2.4 seconds 😭
* **Rows Evaluated:** 2,000,000 rows
* **Engine Status (`EXPLAIN` type):** `ALL`
* **Query stats:**
> ![image_1784049137318.png](images_SQL_NB1/image_1784049137318.png "image_1784049137318.png")
* **Execution Plan:**
> ![image_1784049163673.png](images_SQL_NB1/image_1784049163673.png "image_1784049163673.png")